In [32]:
pip install pandas numpy matplotlib scikit-learn 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [42]:

"""
Reads a CSV with columns:
    population, commercial_activity_score, distance_to_city_center_km,
day_of_week, time_of_day, weather, passenger_demand

Trains both a Multiple Linear Regression (OLS) model and an
Artificial Neural Network (
    feedforward, 1 hidden layer of 5 neurons,
per Heaton's 2008 rule of thumb
), then compares using
RMSE, MAE, MAPE, and R^2 -- matching section 3.8 of the methodology.
"""

from itertools import islice

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor


 1. Load data

In [43]:
CSV_PATH = r"d:\TRISHA\RESEARCH\tricycle_passenger_demand_template.csv"
df = pd.read_csv(CSV_PATH)

FEATURES_NUMERIC = ["population", "commercial_activity_score", "distance_to_city_center_km"]
FEATURES_CATEGORICAL = ["day_of_week", "time_of_day", "weather"]
TARGET = "passenger_demand"
 
X = df[FEATURES_NUMERIC + FEATURES_CATEGORICAL]
y = df[TARGET]
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
 
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), FEATURES_NUMERIC),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), FEATURES_CATEGORICAL),
])
 
mlr_pipeline = Pipeline([("preprocess", preprocessor), ("model", LinearRegression())])
ann_pipeline = Pipeline([("preprocess", preprocessor), ("model", MLPRegressor(
    hidden_layer_sizes=(5,), activation="relu", solver="adam", max_iter=2000, random_state=42,
))])
 
mlr_pipeline.fit(X_train, y_train)
ann_pipeline.fit(X_train, y_train)

C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['population','commercial_activity_score','distance_to_city_center_km', 'day_of_week','time_of_day','weather']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically pas

FIGURE 7 style: forecast overlay across all observations#

In [44]:
y_all_actual = y.reset_index(drop=True)
mlr_all_pred = mlr_pipeline.predict(X)
ann_all_pred = ann_pipeline.predict(X)
obs_index = range(1, len(y_all_actual) + 1)
 
plt.figure(figsize=(12, 5))
plt.plot(obs_index, y_all_actual, color="red", linewidth=1, label="Total Demand")
plt.plot(obs_index, mlr_all_pred, color="#7fbf3f", linewidth=1, label="MLR Forecasting")
plt.plot(obs_index, ann_all_pred, color="#1f3864", linewidth=1, label="ANN Forecasting")
plt.xlabel("Observation")
plt.ylabel("Passenger Demand")
plt.title("Comparison of MLR and ANN Forecasts of Tricycle Passenger Demand")
plt.legend(loc="upper right", ncol=3)
plt.tight_layout()
plt.savefig("figure7_style_forecast_overlay.png", dpi=150)
plt.close()


C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


FIGURE 8 style: fit scatter plots with trend line + R^2, per model

In [46]:
mlr_test_pred = mlr_pipeline.predict(X_test)
ann_test_pred = ann_pipeline.predict(X_test)
 
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
 
for ax, pred, name, color in [
    (axes[0], mlr_test_pred, "MLR", "tab:blue"),
    (axes[1], ann_test_pred, "ANN", "tab:orange"),
]:
    actual = y_test.values
    ax.scatter(actual, pred, color=color, edgecolor="k", alpha=0.8)
 
    slope, intercept, r_value, p_value, std_err = linregress(actual, pred)
    line_x = np.array([actual.min(), actual.max()])
    line_y = slope * line_x + intercept
    ax.plot(line_x, line_y, color="red", linewidth=1.5)
 
    sign = "+" if intercept >= 0 else "-"
    equation = f"y = {slope:.4f}x {sign} {abs(intercept):.4f}"
    r2_text = f"$r^2$ = {r_value**2:.4f}"
    ax.text(0.05, 0.90, equation, transform=ax.transAxes, fontsize=9)
    ax.text(0.05, 0.83, r2_text, transform=ax.transAxes, fontsize=9)
 
    ax.set_xlabel("Actual Demand")
    ax.set_ylabel("Predicted Demand")
    ax.set_title(f"{name} Model Fit")
 
plt.suptitle("Figure 8 Style: Model Fit Comparison for Passenger Demand Estimation", y=1.03)
plt.tight_layout()
plt.savefig("figure8_style_model_fit.png", dpi=150, bbox_inches="tight")
plt.close()
 
print("Saved: figure7_style_forecast_overlay.png and figure8_style_model_fit.png")

C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Saved: figure7_style_forecast_overlay.png and figure8_style_model_fit.png


# 4. Build models

In [37]:
mlr_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression()),
])

ann_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", MLPRegressor(
        hidden_layer_sizes=(5,),   # 1 hidden layer, 5 neurons -- Heaton (2008)
        activation="relu",
        solver="adam",
        max_iter=2000,
        random_state=42,
    )),
])

5. Train

In [38]:
mlr_pipeline.fit(X_train, y_train)
ann_pipeline.fit(X_train, y_train)

C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['population','commercial_activity_score','distance_to_city_center_km', 'day_of_week','time_of_day','weather']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically pas

6. Evaluate: RMSE, MAE, MAPE, R^2 (section 3.8)

In [48]:
def evaluate(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    r2 = r2_score(y_test, y_pred)
    return {"Model": name, "RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2": r2}

results = pd.DataFrame([
    evaluate(mlr_pipeline, X_test, y_test, "MLR"),
    evaluate(ann_pipeline, X_test, y_test, "ANN"),
])

print(results.to_string(index=False))


Model      RMSE       MAE  MAPE (%)          R2
  MLR 32.066619 26.893100 27.988554  -40.130723
  ANN 74.674862 74.625934 80.632255 -222.053399


C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\~Angel~\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


# 7. Inspect MLR coefficients for interpretability

In [40]:
feature_names = mlr_pipeline.named_steps["preprocess"].get_feature_names_out()
coefficients = mlr_pipeline.named_steps["model"].coef_
print("\nMLR coefficients:")
for name, coef in zip(feature_names, coefficients):
    print(f"  {name}: {coef:.4f}")


MLR coefficients:
  num__population: 37.7470
  num__commercial_activity_score: 9.2199
  num__distance_to_city_center_km: -11.6453
  cat__day_of_week_Saturday: 9.6938
  cat__day_of_week_Thursday: 0.6350
  cat__day_of_week_Tuesday: -24.3062
  cat__day_of_week_Wednesday: 18.2541
  cat__time_of_day_Evening: 2.6418
  cat__time_of_day_Morning: -10.3582
  cat__weather_Sunny: 21.6938


STEP 6: Predicted vs Actual plot

In [47]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolor="k")

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Perfect prediction")

plt.xlabel("Actual Passenger Demand")
plt.ylabel("Predicted Passenger Demand")
plt.title(f"Predicted vs Actual Passenger Demand\nR² = {r2:.3f}, MAE = {mae:.2f}")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'y_pred' is not defined

In [49]:
df.plot(kind='scatter', x='col_1', y='col_2')

KeyError: 'col_1'